# Different Chunking Strategies


1. Fixed-size Chunking : Text splitting with a specific chunk size and overlap is optional chunk overlap. This approach is most common and straightforward 

2. Recursive Chunking : Iterating default seperators until one of them produces the preferred chunk size. Default seperators include ["\n\n", "\n", " ", ""].This chunking method follows hierarchical separators so that paragraphs, followed by sentences and then words, are kept together as much as possible.

3. Semantic Chunking : Splitting text in a way that groups sentences based on the semantic similarity of the embeddings. Embeddings of high semantic similarity are closer together than those with the low sematic similarity. 

4. Document based Chunking : Splitting based on the document structure. This splitter can utilize Markdown text, image, 
5. Agentic Chunking

In [29]:
import os
os.chdir("/Users/kavisanthoshkumar/Documents/GenerativeAIusingAWS/HR_Policy_QueryResolutionWithRAG")

# Decorators
from typing import List, Dict
from IPython.display import display, Markdown

from dotenv import load_dotenv


import yaml
with open('config.yaml', 'rb') as file:
    config = yaml.safe_load(file)

import pickle

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import (CharacterTextSplitter, 
                                      RecursiveCharacterTextSplitter)

# embeddings package
from langchain_aws import BedrockEmbeddings
from langchain_pinecone import PineconeVectorStore

# Retrievel Chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)
from langchain_classic.chains import create_retrieval_chain


# Initialize ChatBedrockConverse
from langchain_aws import ChatBedrockConverse


In [2]:
def DocumentViewer(docs:List[str], doc_page:int) -> None:
    print("=== Viewing: Page Content === \n")
    display(Markdown(docs[doc_page].page_content))
    print("=== Viewing: Metadata Information === \n")
    print( docs[doc_page].metadata)

### Validate AWS BedrockEmbeddings 

In [3]:
# Initialize AWS BedRock Titan Embeddings
embeddings = BedrockEmbeddings(
    credentials_profile_name = 'default', 
    region_name = 'us-east-1', 
    model_id = 'cohere.embed-english-v3', 
    dimensions = 1024
)


# Let's create a sample documents
documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    )
]
documents


[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        ')]

In [4]:
texts = [doc.page_content for doc in documents]
document_embeddings = embeddings.embed_documents(texts)

### 1. Read HR Policy Documents

In [5]:
# Loading the hr policy chunked documents 
with open(config['outputs']['chunked_output_file_path'], 'rb') as file:
    hr_policy_chunked_documents = pickle.load(file)
    
print(f"SUCCESS : LOADED HR POLICY CHUNKED DOCUMENTS : {len(hr_policy_chunked_documents)}")

SUCCESS : LOADED HR POLICY CHUNKED DOCUMENTS : 22


### 2. Chunking the Documents 

#### Let's perform semantic chunking 

- To perform semantic chunking we need to have a embedding model
- We are embedding model as AWS Bedrock - Nova Embeddings

In [6]:
text_splitter = SemanticChunker(
    embeddings = embeddings
)
semantic_chunked_docs = text_splitter.split_documents(hr_policy_chunked_documents)

print(f"length of the semantic chunked docs : len(semantic_chunked_docs)")
with open(config['outputs']['semantic_chunked_output_file_path'], 'wb') as file:
    pickle.dump(semantic_chunked_docs, file)

print("SUCCESS == CHUNKED DOCUMENTS USING SEMANTIC CHUNKER AND AWS COHERE EMBEDDINGS")


length of the semantic chunked docs : len(semantic_chunked_docs)
SUCCESS == CHUNKED DOCUMENTS USING SEMANTIC CHUNKER AND AWS COHERE EMBEDDINGS


### 3. SETUP PINECONE FROM THE AWS MARKETPLACE

In [8]:
DocumentViewer(semantic_chunked_docs, 7)

=== Viewing: Page Content === 



Refrain from any 
action that may prevent, restrict, or alter fair competition. Do not 
compromise our firm's reputation or your own by engaging or appearing 
to engage in any form of corruption or misconduct. Never give, offer, 
promise, solicit, or accept anything of value - whether directly or indirectly 
through others such as third-party intermediaries - if it is intended or 
could be perceived as intended to improperly influence decisions on 
behalf of our firm. Be alert and escalate activities that are designed to 
hinder or prevent the detection of improper or illegal activity (e.g., money 
laundering, tax evasion, bribery, etc.). Take ownership - anticipate, identify, and manage the risk and impacts of 
the decisions and actions you take at work. You are in charge of your 
decisions. No one has the authority to tell you to do something unethical 
or illegal. Code of Conduct 
4

=== Viewing: Metadata Information === 

{'producer': 'Adobe Acrobat Pro 2020 20 Paper Capture Plug-in with ClearScan', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-07-23T16:14:08-04:00', 'source': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'file_path': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'total_pages': 18, 'format': 'PDF 1.7', 'title': 'Our Code of Conduct 2024', 'author': 'JP Morgan Chase', 'subject': 'Code of Conduct', 'keywords': 'Our, Code, of, Conduct, 2024, JP, Morgan, Chase', 'moddate': '2024-08-05T10:50:00-04:00', 'trapped': '', 'modDate': "D:20240805105000-04'00'", 'creationDate': "D:20240723161408-04'00'", 'page': 6}


In [51]:
# Creating Index in Pinecone
index_name = 'hrpolicyindex'

vectorstore_from_documents = PineconeVectorStore(
    embedding = embeddings, 
    index_name = index_name
)

print("=== SUCCESS : UPSERTED THE RECORDS ===")

=== SUCCESS : UPSERTED THE RECORDS ===


In [56]:
vectorstore_from_documents.add_documents(documents = semantic_chunked_docs)

retriever = vectorstore_from_documents.as_retriever(
        search_type = 'mmr', 
        search_kwargs = {'k': 3}
    )

In [58]:
retriever.invoke( "Can you tell me about this document?")

[Document(metadata={'author': 'JP Morgan Chase', 'creationDate': "D:20240723161408-04'00'", 'creationdate': '2024-07-23T16:14:08-04:00', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'file_path': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'format': 'PDF 1.7', 'keywords': 'Our, Code, of, Conduct, 2024, JP, Morgan, Chase', 'modDate': "D:20240805105000-04'00'", 'moddate': '2024-08-05T10:50:00-04:00', 'page': 8.0, 'producer': 'Adobe Acrobat Pro 2020 20 Paper Capture Plug-in with ClearScan', 'source': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'subject': 'Code of Conduct', 'title': 'Our Code of Conduct 2024', 'total_pages': 18.0, 'trapped': ''}, page_content='Use it! Contents \nAbout \nReport \n..'),
 Document(metadata={'author': 'JP Morgan Chase', 'creationDate': "D:20240723161408-04'00'", 'creationdate': '2024-07-23T16:14:08-04:00', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'file_path': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf',

In [24]:
nova_lite_model = ChatBedrockConverse(
    model_id = 'amazon.nova-lite-v1:0',   
)

message = [{
    'role': 'user', 
    'content': [{'text': "How are you doing today?"}]
}
]

nova_lite_model.invoke(message)

AIMessage(content="I'm doing well, thank you for asking! How can I assist you today? If you have any questions or need help with something, feel free to let me know.", additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '6f584497-0525-4956-82c9-c9d6738feef1', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 08 Jun 2026 22:56:06 GMT', 'content-type': 'application/json', 'content-length': '347', 'connection': 'keep-alive', 'x-amzn-requestid': '6f584497-0525-4956-82c9-c9d6738feef1'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [456]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'}, id='lc_run--019ea973-4a16-74c0-bb77-1d00a25fe659-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 37, 'total_tokens': 43, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [61]:
def ask(question : str) -> str:
    load_dotenv()

    # Initialize BedrockEmbeddings
    embeddings = BedrockEmbeddings(
        credentials_profile_name = 'default', 
        region_name = 'us-east-1', 
        model_id = 'cohere.embed-english-v3', 
        dimensions = 1024
    )
    
    if not config['indexes']['pinecone_index_name']:
        raise ValueError("Pinecone Index needs to be setup before adding the documents")


    # Search the relevant documents
    docsearch = PineconeVectorStore(
        index_name = config['indexes']['pinecone_index_name'], 
        embedding = embeddings, 
        namespace = '__default__'
    )

    # docsearch as retriever
    retriever = docsearch.as_retriever(
        search_type = 'mmr', 
        search_kwargs = {'k': 3}
    )

    print(config['indexes']['pinecone_index_name'], retriever.invoke(question))

    # Initialize the Amazon Nova lite model
    nova_lite_model = ChatBedrockConverse(
        model_id = 'amazon.nova-lite-v1:0',   
    )

    # Create Prompt Template
    prompt = PromptTemplate(
                    template="""
                You are an expert assistant.

                Instructions:
                - Answer only from the provided context.
                - Do not make up information.
                - If the context does not contain the answer, say:
                "The provided documents do not contain sufficient information."
                - Keep answers concise and factual.
                - When possible, reference the relevant section of the context.

                Retrieved Context:
                {context}

                User Question:
                {input}

                Answer:
                """,
                    input_variables=["context", "input"])

    # Create Stuff Document chain
    combine_docs_chain = create_stuff_documents_chain(
        nova_lite_model, 
        prompt
    )

    # Create Retrievel chain
    retriever_chain = create_retrieval_chain(retriever, 
                                        combine_docs_chain)

    # Generate relevant documents
    result = retriever_chain.invoke({'input':question})

    return result

In [62]:
ask(question =  "Can you tell me about this document?")

hrpolicyindex [Document(metadata={'author': 'JP Morgan Chase', 'creationDate': "D:20240723161408-04'00'", 'creationdate': '2024-07-23T16:14:08-04:00', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'file_path': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'format': 'PDF 1.7', 'keywords': 'Our, Code, of, Conduct, 2024, JP, Morgan, Chase', 'modDate': "D:20240805105000-04'00'", 'moddate': '2024-08-05T10:50:00-04:00', 'page': 8.0, 'producer': 'Adobe Acrobat Pro 2020 20 Paper Capture Plug-in with ClearScan', 'source': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'subject': 'Code of Conduct', 'title': 'Our Code of Conduct 2024', 'total_pages': 18.0, 'trapped': ''}, page_content='Use it! Contents \nAbout \nReport \n..'), Document(metadata={'author': 'JP Morgan Chase', 'creationDate': "D:20240723161408-04'00'", 'creationdate': '2024-07-23T16:14:08-04:00', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'file_path': 'app/input/jp-morgan-chase-code-of-conduct

{'input': 'Can you tell me about this document?',
 'context': [Document(metadata={'author': 'JP Morgan Chase', 'creationDate': "D:20240723161408-04'00'", 'creationdate': '2024-07-23T16:14:08-04:00', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'file_path': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'format': 'PDF 1.7', 'keywords': 'Our, Code, of, Conduct, 2024, JP, Morgan, Chase', 'modDate': "D:20240805105000-04'00'", 'moddate': '2024-08-05T10:50:00-04:00', 'page': 8.0, 'producer': 'Adobe Acrobat Pro 2020 20 Paper Capture Plug-in with ClearScan', 'source': 'app/input/jp-morgan-chase-code-of-conduct-policy.pdf', 'subject': 'Code of Conduct', 'title': 'Our Code of Conduct 2024', 'total_pages': 18.0, 'trapped': ''}, page_content='Use it! Contents \nAbout \nReport \n..'),
  Document(metadata={'author': 'JP Morgan Chase', 'creationDate': "D:20240723161408-04'00'", 'creationdate': '2024-07-23T16:14:08-04:00', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'file